# Exercise 1 solution

## 1.a
By noticing that 
$$x_k = A^k x_0 + A^{k-1} B u_0 + \dots AB u_{k-2} + B u_{k-1}$$

One can write as follows:
$$
\begin{bmatrix}
x_0\\
x_1\\
\vdots\\
x_N
\end{bmatrix}
=
\begin{bmatrix}
I\\
A\\
\vdots\\
A^{N}
\end{bmatrix}x(0)
+
\begin{bmatrix}
0 & \cdots & \cdots & 0\\
B & 0      & \cdots & 0\\
AB & B     & \ddots & \vdots\\
\vdots & \ddots & \ddots & 0\\
A^{N-1}B & \cdots & AB & B
\end{bmatrix}
\begin{bmatrix}
u_0\\
u_1\\
\vdots\\
u_{N-1}
\end{bmatrix}
$$

Or in a more compact way as :
$$ X = P_X x_0 + P_U U $$

And thus getting the following expression for the cost 
$$ J = X^T \bar{Q} X + U^T \bar{R} U $$
Where $\bar{Q}=\text{blockdiag}(Q,\dots,Q,S) $ and $\bar{R} = \text{blockdiag} (R,\dots,R)$

By sobsituting the expression of $X$ into $J$ it is easy to get that
$$ J = U^T F U + 2f^T U + c$$
Where $F = P^T_U \bar{Q} P_U + \bar{R} $ , $f^T = x_0 ^T P_X^T \bar{Q} P_U$ and $c = x_0^T P_X^T \bar{Q} P_X x_0$

## 1.b
Since the problem is uncostrained and J is positive definite we can set the gradient to 0 and get
$$ U* = - F^{-1} f^T = -(P^T_U \bar{Q} P_U + \bar{R})^{-1} x_0^T P^T_X \bar{Q} P_X

## 1.c

In [4]:
import numpy as np

# define given matrix
A = np.array([[0.88, 0.2, 0.49], [1.12, 0.94, -0.49], [0.48, -0.08, -0.05]])
B = np.array([[0.28 , 0.36] , [0.3 , 0.27] , [0.21 , 0.33]])

Q = np.diag([1, 1, 1])
R = np.diag([2, 2])
S = np.diag([0.5, 0.5, 0.5])

T = 10  # time horizon
n = 3  # state dimension
m = 2  # control dimension
x0 = np.array([1, 0, -1])  # initial state

A_powers = [np.linalg.matrix_power(A, i) for i in range(T)]
P_X = np.concatenate([np.identity(n) , np.vstack(A_powers)] , axis=0)
P_U = np.zeros(((T+1) * n, T * m))
# Compute optimal control using equations derived above
for i in range(0, (T+1)*n, n): 
    for j in range(0, T*m, m):
        if j//m < i//n:
            P_U[i:i+n,j:j+m] = np.dot(A_powers[i//n - j//m - 1], B)
        else:
            P_U[i:i+n,j:j+m] = np.zeros((n, m))

# Build the cost matrices
Q_bar = np.zeros(((T+1)*n, (T+1)*n))
R_bar = np.zeros((T*m, T*m))

for i in range(0, (T+1)*n, n):
    if i < T*n:
        Q_bar[i:i+n, i:i+n] = Q
    else:        
        Q_bar[i:i+n, i:i+n] = S

for i in range(0, T*m, m):
    R_bar[i:i+m, i:i+m] = R

# Compute matrices F and f^T

F = np.dot(P_U.T, np.dot(Q_bar, P_U)) + R_bar
f_T = np.dot(x0.T, np.dot(P_X.T, np.dot(Q_bar, P_U)))

# Solve for optimal control sequence
U_opt = -np.dot(np.linalg.inv(F), f_T)


ModuleNotFoundError: No module named 'numpy'

In [3]:
# Solve using CVXPY
import cvxpy as cp

# Decision variables
x = cp.Variable((T + 1, n))  # states x_0, ..., x_T
u = cp.Variable((T, m))       # controls u_0, ..., u_{T-1}

cost = 0
constraints = [x[0] == x0]

for k in range(T):
    cost += cp.quad_form(x[k], Q) + cp.quad_form(u[k], R)
    constraints += [x[k + 1] == A @ x[k] + B @ u[k]]

# terminal cost
cost += cp.quad_form(x[T], S)

prob = cp.Problem(cp.Minimize(cost), constraints)
prob.solve()

print(f"Status: {prob.status}")
print(f"Optimal cost (CVXPY):      {prob.value:.6f}")

# Compare with analytical solution
U_opt_mat = U_opt.reshape(T, m)
X_opt = P_X @ x0 + P_U @ U_opt
J_analytical = X_opt @ Q_bar @ X_opt + U_opt @ R_bar @ U_opt
print(f"Optimal cost (analytical): {J_analytical:.6f}")

print(f"\nMax difference in u: {np.max(np.abs(u.value - U_opt_mat)):.2e}")

ModuleNotFoundError: No module named 'cvxpy'

In [ ]:
import matplotlib.pyplot as plt

U_cvxpy = u.value          # (T, m)
U_analytic = U_opt.reshape(T, m)

# Reconstruct state trajectories
X_cvxpy = x.value          # (T+1, n)

X_analytic = np.zeros((T + 1, n))
X_analytic[0] = x0
for k in range(T):
    X_analytic[k + 1] = A @ X_analytic[k] + B @ U_analytic[k]

time_u = np.arange(T)
time_x = np.arange(T + 1)

fig, axes = plt.subplots(3, 1, figsize=(10, 12))

# --- Control inputs ---
ax = axes[0]
for i in range(m):
    ax.plot(time_u, U_analytic[:, i], label=f"u{i+1} analytical", linestyle="-")
    ax.plot(time_u, U_cvxpy[:, i],   label=f"u{i+1} CVXPY",      linestyle="--")
ax.set_title("Optimal Control Inputs")
ax.set_xlabel("Time step k")
ax.set_ylabel("u")
ax.legend()
ax.grid(True)

# --- State trajectories ---
ax = axes[1]
for i in range(n):
    ax.plot(time_x, X_analytic[:, i], label=f"x{i+1} analytical", linestyle="-")
    ax.plot(time_x, X_cvxpy[:, i],    label=f"x{i+1} CVXPY",      linestyle="--")
ax.set_title("State Trajectories")
ax.set_xlabel("Time step k")
ax.set_ylabel("x")
ax.legend()
ax.grid(True)

# --- Absolute differences ---
ax = axes[2]
for i in range(m):
    ax.plot(time_u, np.abs(U_analytic[:, i] - U_cvxpy[:, i]), label=f"|Δu{i+1}|")
for i in range(n):
    ax.plot(time_x, np.abs(X_analytic[:, i] - X_cvxpy[:, i]), label=f"|Δx{i+1}|", linestyle="--")
ax.set_title("Absolute Difference (Analytical vs CVXPY)")
ax.set_xlabel("Time step k")
ax.set_ylabel("Absolute error")
ax.legend()
ax.grid(True)
ax.set_yscale("log")

plt.tight_layout()
plt.show()
